In [1]:
import kagglehub
import pandas as pd
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import heapq

import string
import spacy
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')

c:\Users\kasih\nlp_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kasih\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kasih\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kasih\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [2]:
import gensim
import pickle
from scipy.sparse import save_npz
from gensim.models.phrases import Phraser, Phrases
from gensim.models.word2vec import Word2Vec

Load Data

In [3]:
csv_file_path = '../data/goemotions_1.csv'
df = pd.read_csv(csv_file_path)
display(df.head())

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [4]:
corpus=df['text'].tolist()
print(f"Corpus loaded: {len(corpus)} documents")
print(f"First document ({len(corpus[0].split())} words):\n{corpus[0][:150]}...")

Corpus loaded: 70000 documents
First document (3 words):
That game hurt....


Text Preprocessing

In [5]:
# convert to lowercase and tokenize 
lower_corpus = [i.lower() for i in corpus]
stop_words = set(stopwords.words('english'))
words = [word_tokenize(i) for i in lower_corpus]

#remove stopwords and special chars per document then lemmatize 
lem = WordNetLemmatizer()
cleaned = []
cleaned_words=[]
for sentence in words:
  cleaned_tokens=[]
  for token in sentence:
        cleaned_token = token.strip(string.punctuation)
        if cleaned_token and cleaned_token not in stop_words and cleaned_token.isalpha():
            lemmatized = lem.lemmatize(cleaned_token)
            cleaned_tokens.append(lemmatized)
  cleaned.append(" ".join(cleaned_tokens)) 
  cleaned_words.append(cleaned_tokens) 

print("Example of cleaned tokens:",cleaned_words)
print(f"\nAfter join as sentences: {cleaned}")

Example of cleaned tokens: [['game', 'hurt'], ['sexuality', 'grouping', 'category', 'make', 'different', 'othet', 'ppl', 'imo', 'fit', 'definition', 'grouping'], ['right', 'care', 'fuck', 'em'], ['man', 'love', 'reddit'], ['name', 'nowhere', 'near', 'falcon'], ['right', 'considering', 'important', 'document', 'know', 'damned', 'thing', 'backwards', 'forward', 'thanks', 'help'], ['big', 'still', 'quite', 'popular', 'heard', 'thing', 'content', 'never', 'watched', 'much'], ['crazy', 'went', 'super', 'religion', 'high', 'school', 'think', 'remember', 'girl', 'entire', 'year', 'became', 'teen', 'mom'], ['adorable', 'asf'], ['sponge', 'blurb', 'pub', 'quaw', 'haha', 'gurr', 'ha', 'aaa', 'finale', 'real'], ['mention', 'think', 'triggered', 'nostalgia'], ['wanted', 'downvote', 'fault', 'homie'], ['turn'], ['odd'], ['build', 'wall', 'jk'], ['appreciate', 'good', 'know', 'hope', 'apply', 'knowledge', 'one', 'day'], ['one', 'time', 'stopped', 'right', 'able', 'get', 'good', 'photo', 'platform', 

cleaned_words : use for w2v model





cleaned : use for tfidf

In [6]:
with open('../data/cleaned_words.pkl', 'wb') as f:
    pickle.dump(cleaned_words, f)

Feature Extraction W2V SkipGram

In [7]:
skipgram = Word2Vec(
    sentences=cleaned_words,
    vector_size=64,
    sg=1,
    window=10,
    epochs=10,
    min_count=10,
    workers=4,
    seed=42
)
skipgram.save("../models/word2vec.model")

print(f"Skip-gram vocab: {len(skipgram.wv)}")
print(f"Successfully saved word2vec model.")

Skip-gram vocab: 4882
Successfully saved word2vec model.


Feature Extraction Tfidf

In [8]:
tfidf=TfidfVectorizer(max_features=500, norm='l2')
tfidf_matrix=tfidf.fit_transform((cleaned))

In [9]:
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

save_npz('../data/tfidf_vectors.npz', tfidf_matrix)

print(f"Successfully saved vectorizer object in.")
print(f"Successfully saved tfidf matrix.")

Successfully saved vectorizer object in.
Successfully saved tfidf matrix.
